# Análisis de Clustering - Segmentación de Entregas

## Objetivo

Aplicar técnicas de clustering (K-Means) para descubrir patrones y segmentar entregas en grupos con características operativas similares. Este análisis nos permite identificar perfiles de entregas y optimizar estrategias logísticas según cada segmento.

## Metodología

### ¿Por qué K-Means?

A diferencia de los modelos supervisados, el clustering no supervisado no requiere una variable objetivo. Nuestro algoritmo busca patrones y agrupaciones naturales en los datos de entregas basándose en características operativas.

### Preparación de datos

Para garantizar un análisis justo donde variables con escalas grandes no dominen los cálculos de distancia:
- **Variables numéricas**: Aplicamos `StandardScaler` para normalizar a media=0 y desv.est.=1
- **Variables categóricas**: Transformamos con `OneHotEncoder` a representación numérica

### Selección del número óptimo de clusters

Evaluaremos múltiples valores de k (entre 2 y 8) usando cuatro métricas complementarias:
- **Inertia (Método del codo)**: Suma de distancias al centroide más cercano
- **Silhouette Score**: Mide cohesión y separación de clusters (rango: -1 a 1)
- **Calinski-Harabasz**: Ratio entre separación e inercia (mayor es mejor)
- **Davies-Bouldin**: Distancia promedio entre clusters (menor es mejor)

## 1. Configuración e Importación de Datos

### Paso 1.1: Cargar módulos necesarios

In [ ]:
import importlib
import pandas as pd
from src import unsupervised_modeling as um
importlib.reload(um)

Importamos las librerías necesarias y recargamos el módulo de clustering para asegurar que tengamos las funciones más actualizadas.

In [ ]:
DATA_PATH = "../data/processed/logistica_clean.csv"
TARGET = "target_tiempo_entrega"

df = pd.read_csv(DATA_PATH)
df.head()

### Paso 1.2: Cargar y explorar los datos procesados

Cargamos el conjunto de datos ya limpio y procesado. Este dataset contiene todas las características operativas de nuestras entregas.

In [ ]:
df.shape

Revisamos las dimensiones del dataset (filas y columnas):

In [ ]:
df.info()

Información detallada del dataset: tipos de datos y valores faltantes:

## 2. Preparación de Datos para Clustering

### Paso 2.1: Separar variables numéricas y categóricas

Dividimos las características en dos grupos para aplicar transformaciones específicas a cada tipo:

In [ ]:
X, numeric_features, categorical_features = um.split_features_for_clustering(
    df,
    target_column=TARGET
)
numeric_features, categorical_features

In [ ]:
preprocessor = um.build_preprocessor(numeric_features, categorical_features)

X_processed = preprocessor.fit_transform(X)

X_processed.shape

### Paso 2.2: Normalizar y codificar variables

Aplicamos `StandardScaler` para variables numéricas y `OneHotEncoder` para categóricas. Esto asegura que todas las variables contribuyan equitativamente al algoritmo de clustering:

## 3. Selección del Número Óptimo de Clusters

### Paso 3.1: Evaluar diferentes números de clusters

Ejecutamos K-Means con valores de k entre 2 y 8, calculando las cuatro métricas para cada configuración:

In [ ]:
metrics_df = um.evaluate_kmeans_range(
    X_processed,
    k_min=2,
    k_max=8
)

metrics_df

In [ ]:
um.plot_elbow(metrics_df)
um.plot_silhouette(metrics_df)

### Paso 3.2: Visualizar las métricas de selección

Graficamos el método del codo (inertia vs k) y el Silhouette Score para identificar el valor óptimo de clusters. Buscaremos el punto donde las métricas se estabilizan o muestran cambios significativos:

## 4. Entrenamiento del Modelo Final

### Paso 4.1: Entrenar K-Means con el número óptimo de clusters

Basándonos en los análisis anteriores, seleccionamos el valor de k más adecuado y entrenamos el modelo final. Ajusta `best_k` según los resultados de las métricas y criterios de negocio:

In [ ]:
best_k = 3  # Ajustar segun metricas y criterio de negocio\n
kmeans_model, labels = um.train_kmeans(
    X_processed,
    n_clusters=best_k
)

In [ ]:
df_clustered, cluster_profile = um.create_cluster_profile(
    df_original=df,
    labels=labels,
    target_column=TARGET
)

cluster_profile

### Paso 4.2: Crear perfil de clusters

Generamos un resumen estadístico de cada cluster. Esto nos ayuda a entender las características típicas de cada segmento de entregas:

In [ ]:
um.plot_clusters_pca(
    X_processed=X_processed,
    labels=labels
)

### Paso 4.3: Visualizar clusters en 2D

Utilizamos PCA (Principal Component Analysis) para reducir dimensionalidad y visualizar los clusters. Esto nos permite ver la distribución y separación de los segmentos:

In [ ]:
um.save_clustering_outputs(
    metrics_df=metrics_df,
    profile=cluster_profile,
    model=kmeans_model,
    output_metrics_path="../results/metrics/clustering_metrics.csv",
    output_profile_path="../results/reports/cluster_profile.csv",
    output_model_path="../models/trained_models/kmeans_model.pkl"
)

### Paso 4.4: Guardar resultados

Almacenamos todos los resultados (métricas, perfil de clusters y modelo entrenado) para su uso posterior en análisis o predicciones:

## 5. Interpretación de Resultados

### ¿Qué nos cuentan los clusters?

Los clusters descubiertos nos permiten segmentar las entregas en grupos con comportamientos operativos similares. Cada cluster representa un perfil de entrega característico definido por variables como:

- **Distancia**: Corta, media o larga
- **Tráfico y clima**: Condiciones viales y meteorológicas
- **Características del vehículo**: Tipo y capacidad
- **Carga**: Peso y volumen de los paquetes
- **Paradas previas**: Número de entregas antes de esta
- **Costos**: Costo de envío y consumo de combustible
- **Tiempo de entrega**: Duración real de la ruta

### Perfiles de entregas identificados

Típicamente encontramos segmentos como:

1. **Entregas Simples (Short & Light)**: Corta distancia, bajo peso, sin paradas previas. Tiempo de entrega mínimo.
2. **Entregas Complejas (Multi-Stop)**: Múltiples paradas, mayor costo operativo, tráfico moderado.
3. **Entregas de Largo Recorrido (Long & Heavy)**: Mayor distancia, mayor peso, alto consumo de combustible.